In [1]:
# Analyze VMM measures specifically
print("\nVisual Meaning Making (VMM) Task Analysis:")
print("=" * 50)

# Calculate means by condition
vmm_vars = ['vmm_meaning', 'vmm_difficulty', 'vmm_timing']
for var in vmm_vars:
    means = df.groupby('condition')[var].agg(['mean', 'std', 'count'])
    print(f"\n{var} by condition:")
    print("-" * 30)
    print(means)

# Test for condition differences
for var in vmm_vars:
    result = stats.ttest_ind(
        df[df['condition'] == 'hopalong'][var],
        df[df['condition'] == 'factory'][var]
    )
    print(f"\nt-test for {var} by condition:")
    print(f"t = {result.statistic:.3f}, p = {result.pvalue:.3f}")

# Correlations with other measures
corr_vars = ['tas', 'stai_post', 'mlq_post', 'wcs_post']
for vmm_var in vmm_vars:
    print(f"\nCorrelations with {vmm_var}:")
    print("-" * 30)
    for var in corr_vars:
        corr = stats.pearsonr(df[vmm_var], df[var])
        print(f"{var}: r = {corr[0]:.3f}, p = {corr[1]:.3f}")


Visual Meaning Making (VMM) Task Analysis:


NameError: name 'df' is not defined

In [ ]:
# Create violin plots for VMM measures
plt.figure(figsize=(15, 5))

for i, var in enumerate(['vmm_meaning', 'vmm_difficulty', 'vmm_timing']):
    plt.subplot(1, 3, i+1)
    sns.violinplot(data=df, x='condition', y=var)
    plt.title(var.replace('vmm_', 'VMM ').title())
    plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('plots/vmm_measures.png', dpi=300, bbox_inches='tight')
plt.close()

## Visual Meaning Making (VMM) Task Results

The VMM task was analyzed across three dimensions:
1. Meaning - How meaningful participants found the visual experience
2. Difficulty - How difficult it was to find meaning in the visuals
3. Timing - Temporal aspects of the meaning-making experience

Significant results and correlations with other measures are shown above.

The plots in 'plots/vmm_measures.png' show the distribution of scores across conditions.

In [ ]:
# Define measures to analyze
measures = {
    'STAI': ['stai_pre', 'stai_post', 'p_stai'],
    'MLQ': ['mlq_pre', 'mlq_post', 'p_mlq'],
    'WCS': ['wcs_pre', 'wcs_post', 'p_wcs'],
    'DAT': ['dat_pre', 'dat_post', 'p_dat']
}

# Analyze each measure
results = {}
for measure_name, columns in measures.items():
    print(f"\nAnalyzing {measure_name}...")
    aov, long_df = analyze_measure(df, measure_name, columns)
    
    if aov is not None:
        results[measure_name] = {
            'anova': aov,
            'data': long_df
        }
        
        # Print ANOVA results
        print(f"\n{measure_name} ANOVA Results:")
        print("-" * 40)
        print(aov)
        
        # Create visualization
        plot_measure_results(long_df, measure_name)

In [ ]:
# Define measures to analyze
measures = {
    'STAI': ['stai_pre', 'stai_post', 'p_stai'],
    'MLQ': ['mlq_pre', 'mlq_post', 'p_mlq'],
    'WCS': ['wcs_pre', 'wcs_post', 'p_wcs'],
    'DAT': ['dat_pre', 'dat_post', 'p_dat'],
    'VMM_Meaning': ['vmm_meaning', 'p_vmm_meaning'],
    'VMM_Difficulty': ['vmm_difficulty', 'p_vmm_difficulty'],
    'VMM_Timing': ['vmm_timing', 'p_vmm_timing']
}

In [ ]:
print("\nSignificant Effects Summary:")
print("=" * 50)

for measure_name, result in results.items():
    aov = result['anova']
    sig_effects = aov[aov['p-unc'] < alpha]
    
    if not sig_effects.empty:
        print(f"\n{measure_name}:")
        print("-" * len(measure_name))
        
        for _, effect in sig_effects.iterrows():
            print(f"\n* {effect['Source']}:")
            print(f"  F({effect['ddof1']}, {effect['ddof2']}) = {effect['F']:.2f}")
            print(f"  p = {effect['p-unc']:.4f}")
            print(f"  partial η² = {effect['np2']:.3f}")
            
            # Post-hoc tests for significant effects
            if effect['Source'] == 'time':
                posthoc = pg.pairwise_ttests(
                    data=result['data'],
                    dv='value',
                    within='time',
                    subject='participant_id'
                )
                sig_posthoc = posthoc[posthoc['p-corr'] < alpha]
                
                if not sig_posthoc.empty:
                    print("\n  Post-hoc comparisons (Bonferroni-corrected):")
                    for _, test in sig_posthoc.iterrows():
                        print(f"    Time {test['A']} vs Time {test['B']}: p = {test['p-corr']:.4f}")

In [ ]:
# Create a summary table of effect sizes
effect_sizes = []
for measure_name, result in results.items():
    aov = result['anova']
    sig_effects = aov[aov['p-unc'] < alpha]
    
    for _, effect in sig_effects.iterrows():
        effect_sizes.append({
            'Measure': measure_name,
            'Effect': effect['Source'],
            'F-value': effect['F'],
            'p-value': effect['p-unc'],
            'partial η²': effect['np2']
        })

if effect_sizes:
    effect_size_df = pd.DataFrame(effect_sizes)
    effect_size_df = effect_size_df.sort_values('partial η²', ascending=False)
    
    print("\nEffect Size Summary (sorted by magnitude):")
    print("=" * 50)
    print(effect_size_df.to_string(index=False, float_format=lambda x: f"{x:.3f}")
)

## Interpretation of Results

Partial η² (eta-squared) effect size interpretations:
- Small effect: ~0.01
- Medium effect: ~0.06
- Large effect: ~0.14

The analysis revealed:

1. **Time Effects**: Changes in measures from pre-test to post-test to follow-up
2. **Condition Effects**: Differences between hopalong and factory conditions
3. **Interaction Effects**: Different patterns of change between conditions

Significant effects are listed above with their F-statistics, p-values, and effect sizes. Post-hoc tests were conducted for significant time effects to identify which specific time points differed from each other.

All visualizations have been saved in the 'plots' directory.

In [ ]:
def prepare_long_data(df, time_columns):
    """Convert data to long format for repeated measures analysis."""
    long_data = []
    for subject in df['participant_id'].unique():
        subject_data = df[df['participant_id'] == subject]
        if not subject_data.empty:
            condition = subject_data['condition'].iloc[0]
            for time_idx, col in enumerate(time_columns):
                if col in subject_data.columns:
                    value = subject_data[col].iloc[0]
                    if not pd.isna(value):
                        long_data.append({
                            'participant_id': subject,
                            'condition': condition,
                            'time': time_idx,
                            'value': value
                        })
    return pd.DataFrame(long_data) if long_data else None

def analyze_measure(df, measure_name, time_columns):
    """Perform repeated measures ANOVA for a given measure."""
    long_df = prepare_long_data(df, time_columns)
    if long_df is None:
        return None, None
        
    try:
        # Perform mixed ANOVA
        aov = pg.mixed_anova(
            data=long_df,
            dv='value',
            within='time',
            between='condition',
            subject='participant_id'
        )
        
        return aov, long_df
    except Exception as e:
        print(f"Error analyzing {measure_name}: {str(e)}")
        return None, None

def plot_measure_results(long_df, measure_name):
    """Create visualization of results."""
    plt.figure(figsize=(10, 6))
    
    # Calculate means and standard errors
    summary = long_df.groupby(['condition', 'time'])['value'].agg(['mean', 'std', 'size']).reset_index()
    summary['se'] = summary['std'] / np.sqrt(summary['size'])
    
    # Plot for each condition
    colors = {'hopalong': '#2ecc71', 'factory': '#e74c3c'}
    for condition in long_df['condition'].unique():
        data = summary[summary['condition'] == condition]
        plt.errorbar(data['time'], data['mean'], yerr=data['se'],
                     label=condition.capitalize(), marker='o',
                     color=colors.get(condition, 'gray'),
                     capsize=5, markersize=8, linewidth=2)
    
    plt.title(f'{measure_name} Results by Condition and Time')
    plt.xlabel('Time Point (0=Pre, 1=Post, 2=Follow-up)')
    plt.ylabel(f'{measure_name} Score')
    plt.legend(title='Condition')
    plt.grid(True, alpha=0.3)
    
    # Add significance annotations if needed
    plt.tight_layout()
    plt.savefig(f'plots/{measure_name.lower()}_rmanova_plot.png', dpi=300, bbox_inches='tight')
    plt.close()

# Repeated Measures ANOVA Analysis

This notebook analyzes the effects of conditions (hopalong vs factory) on various psychological measures across time points using repeated measures ANOVA. We'll examine:

- STAI (State-Trait Anxiety Inventory)
- MLQ (Meaning in Life Questionnaire)
- WCS (Wise Crowds Score)
- DAT (Duration Assessment Test)

The analysis will highlight statistically significant effects (p < 0.05) for:
1. Main effect of condition
2. Main effect of time
3. Condition × time interactions

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pingouin as pg
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn')
sns.set_context('talk')

# Load data
df = pd.read_pickle('data/raw.pkl')

# Define significance threshold
alpha = 0.05

print(f"Loaded data with {len(df)} participants")
print(f"Conditions: {df['condition'].unique()}")

In [ ]:
# This will be an empty notebook that we'll edit next